# Chapitre 6 - Cours 4 : index secondaires et partitionnement

Tant qu'on accède aux données par leur clé, le partitionnement est simple : la clé de
partitionnement désigne la partition, donc le nœud. Mais une vraie application veut aussi
chercher **par valeur** : toutes les voitures rouges, toutes les commandes d'un client,
tous les documents qui contiennent un mot. C'est le rôle des **index secondaires**, et
ils ne se laissent pas découper selon la clé principale. Ce notebook couvre la section
*Partitioning and Secondary Indexes* (sous-sections *Partitioning Secondary Indexes by Document* et *Partitioning Secondary Indexes by Term*).

**Plan.**

1. Le problème : un index secondaire ne suit pas le découpage des données.
2. Les index locaux (partitionnés par document) : écrire sur une partition, lire partout.
3. Le prix du scatter/gather : latence de queue et débit qui ne passe pas à l'échelle.
4. Les index globaux (partitionnés par terme) : lire une partition, écrire sur plusieurs.
5. Comparer et choisir : un modèle de coût et son point de bascule.

In [1]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import partitioning as P
from ddia.diagrams import Diagram
from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 2)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")
SH = "partition"          # vocabulaire de l'édition, pour les figures
SHS = "partitions"
FEM = "e"                              # accord en genre du mot de l'édition